In [ ]:
"""
第2部分：基础检索功能
"""

from elasticsearch import Elasticsearch
import json


class MedicalGuidelineSearcher:
    """医疗指南检索器"""
    
    def __init__(self, es, index_name='medical_guidelines'):
        self.es = es
        self.index_name = index_name
    
    # ==================== 1. 基础搜索 ====================
    def search_by_drug_name(self, drug_name, size=10):
        """按药物名称搜索"""
        
        query = {
            "query": {
                "match": {
                    "intervention.name": {
                        "query": drug_name,
                        "operator": "and"
                    }
                }
            },
            "size": size
        }
        
        result = self.es.search(index=self.index_name, body=query)
        return self._format_results(result)
    
    # ==================== 2. 多条件组合搜索 ====================
    def search_by_conditions(self, drug_name=None, phase=None, 
                            treatment_type=None, category=None, size=10):
        """多条件组合搜索"""
        
        must_clauses = []
        
        if drug_name:
            must_clauses.append({
                "match": {
                    "intervention.name": drug_name
                }
            })
        
        if phase:
            must_clauses.append({
                "term": {
                    "phase": phase
                }
            })
        
        if treatment_type:
            must_clauses.append({
                "term": {
                    "treatment_type": treatment_type
                }
            })
        
        if category:
            must_clauses.append({
                "term": {
                    "intervention.category": category
                }
            })
        
        query = {
            "query": {
                "bool": {
                    "must": must_clauses
                }
            },
            "size": size
        }
        
        result = self.es.search(index=self.index_name, body=query)
        return self._format_results(result)
    
    # ==================== 3. 按治疗阶段搜索 ====================
    def search_acute_treatments(self, size=20):
        """搜索急性期治疗"""
        
        query = {
            "query": {
                "term": {
                    "phase": "急性期"
                }
            },
            "sort": [
                {"evidence.recommendation_grade": "asc"}
            ],
            "size": size
        }
        
        result = self.es.search(index=self.index_name, body=query)
        return self._format_results(result)
    
    def search_prevention_treatments(self, size=20):
        """搜索预防期治疗"""
        
        query = {
            "query": {
                "term": {
                    "phase": "预防期"
                }
            },
            "sort": [
                {"evidence.recommendation_grade": "asc"}
            ],
            "size": size
        }
        
        result = self.es.search(index=self.index_name, body=query)
        return self._format_results(result)
    
    # ==================== 4. 按药物分类搜索 ====================
    def search_by_category(self, category, size=10):
        """按药物分类搜索"""
        
        query = {
            "query": {
                "term": {
                    "intervention.category": category
                }
            },
            "size": size
        }
        
        result = self.es.search(index=self.index_name, body=query)
        return self._format_results(result)
    
    # ==================== 5. 全文检索 ====================
    def full_text_search(self, text, size=10):
        """全文检索（支持自然语言问题）"""
        
        query = {
            "query": {
                "multi_match": {
                    "query": text,
                    "fields": [
                        "intervention.name^3",           # 药物名称权重最高
                        "intervention.category^2",        # 分类权重次之
                        "intervention.route.description",
                        "safety.adverse_effects.common",
                        "safety.contraindications",
                        "traceability.raw_text_segment"
                    ],
                    "type": "best_fields"
                }
            },
            "size": size
        }
        
        result = self.es.search(index=self.index_name, body=query)
        return self._format_results(result)
    
    # ==================== 6. 高级筛选 ====================
    def advanced_filter(self, filters, size=10):
        """高级筛选
        
        示例filters:
        {
            'phase': '急性期',
            'treatment_type': 'drug',
            'evidence_level': '证据级别I',
            'route': 'oral',
            'exclude_pregnancy': True
        }
        """
        
        must_clauses = []
        must_not_clauses = []
        
        # 基础筛选
        if 'phase' in filters:
            must_clauses.append({"term": {"phase": filters['phase']}})
        
        if 'treatment_type' in filters:
            must_clauses.append({"term": {"treatment_type": filters['treatment_type']}})
        
        if 'evidence_level' in filters:
            must_clauses.append({"term": {"evidence.evidence_level": filters['evidence_level']}})
        
        if 'route' in filters:
            must_clauses.append({"term": {"intervention.route.primary": filters['route']}})
        
        if 'category' in filters:
            must_clauses.append({"term": {"intervention.category": filters['category']}})
        
        # 排除妊娠期禁用
        if filters.get('exclude_pregnancy'):
            must_not_clauses.append({
                "match": {
                    "applicability.pregnancy_lactation": "禁用"
                }
            })
        
        query = {
            "query": {
                "bool": {
                    "must": must_clauses,
                    "must_not": must_not_clauses
                }
            },
            "size": size
        }
        
        result = self.es.search(index=self.index_name, body=query)
        return self._format_results(result)
    
    # ==================== 7. 格式化结果 ====================
    def _format_results(self, result):
        """格式化ES返回结果"""
        
        hits = result['hits']['hits']
        total = result['hits']['total']['value']
        
        formatted = {
            'total': total,
            'results': []
        }
        
        for hit in hits:
            source = hit['_source']
            formatted_item = {
                'score': hit['_score'],
                'drug_name': source['intervention']['name'],
                'category': source['intervention'].get('category', ''),
                'phase': source['phase'],
                'treatment_type': source['treatment_type'],
                'route': source['intervention']['route'].get('description', ''),
                'frequency': source['intervention'].get('frequency', ''),
                'evidence_level': source['evidence'].get('evidence_level', ''),
                'recommendation_grade': source['evidence'].get('recommendation_grade', ''),
                'contraindications': source['safety'].get('contraindications', []),
                'adverse_effects': source['safety']['adverse_effects'].get('common', ''),
                'source_guideline': source['source'].get('guideline', ''),
                'source_year': source['source'].get('year', ''),
                'full_doc': source  # 保留完整文档供详细查看
            }
            formatted['results'].append(formatted_item)
        
        return formatted
    
    # ==================== 8. 打印结果 ====================
    def print_results(self, results, show_full=False):
        """打印检索结果"""
        
        print(f"\n找到 {results['total']} 条结果")
        print("="*80)
        
        for i, item in enumerate(results['results'], 1):
            print(f"\n【结果 {i}】 相关度评分: {item['score']:.2f}")
            print(f"药物名称: {item['drug_name']}")
            print(f"药物分类: {item['category']}")
            print(f"治疗阶段: {item['phase']}")
            print(f"给药方式: {item['route']}")
            print(f"用药频次: {item['frequency']}")
            print(f"证据等级: {item['evidence_level']}")
            print(f"推荐强度: {item['recommendation_grade']}")
            
            if item['contraindications']:
                print(f"禁忌症: {', '.join(item['contraindications'][:3])}")
            
            if item['adverse_effects']:
                print(f"不良反应: {item['adverse_effects'][:100]}...")
            
            print(f"来源: {item['source_guideline']} ({item['source_year']})")
            
            if show_full:
                print("\n完整文档:")
                print(json.dumps(item['full_doc'], ensure_ascii=False, indent=2))
            
            print("-"*80)


# ==================== 使用示例 ====================
if __name__ == "__main__":
    # 连接ES
    es = Elasticsearch([{'host': 'localhost', 'port': 9200, 'scheme': 'http'}])
    
    # 创建检索器
    searcher = MedicalGuidelineSearcher(es)
    
    # 示例1: 按药物名称搜索
    print("\n【示例1】搜索'布洛芬':")
    results = searcher.search_by_drug_name("布洛芬")
    searcher.print_results(results)
    
    # 示例2: 搜索急性期治疗
    print("\n【示例2】搜索急性期治疗:")
    results = searcher.search_acute_treatments(size=5)
    searcher.print_results(results)
    
    # 示例3: 多条件搜索
    print("\n【示例3】搜索急性期口服药物:")
    results = searcher.search_by_conditions(
        phase="急性期",
        treatment_type="drug"
    )
    searcher.print_results(results)
    
    # 示例4: 全文检索
    print("\n【示例4】全文检索'偏头痛急性发作吃什么药':")
    results = searcher.full_text_search("偏头痛急性发作吃什么药")
    searcher.print_results(results)
    
    # 示例5: 高级筛选
    print("\n【示例5】高级筛选 - 急性期口服非甾体抗炎药:")
    results = searcher.advanced_filter({
        'phase': '急性期',
        'treatment_type': 'drug',
        'category': '非甾体抗炎药',
        'route': 'oral'
    })
    searcher.print_results(results)